# 05 - 高级特性（Advanced Features）
> Harvard CS50's Intro to Databases with SQL  |  课程时间：5:49:09 – 7:31:17

这一章进入数据库的「进阶工具箱」—— 触发器自动响应事件、视图封装复杂查询、CTE 让 SQL 像写程序、软删除保护数据、多对多关系建模。

## 学习路线

| # | 内容 |
|---|------|
| 5.1 | Trigger — 触发器，自动响应数据变更 |
| 5.2 | View — 视图，封装复杂查询为虚拟表 |
| 5.3 | CTE — 公共表表达式，WITH + 递归查询 |
| 5.4 | Soft Delete — 软删除，标记而非真删 |
| 5.5 | Many-to-Many — 多对多关系 + 联结表 |

> ⚠️ 先运行下面的 cell 创建练习环境！

In [ ]:
import sqlite3

def sql(query):
    statements = [s.strip() for s in query.strip().split(';') if s.strip()]
    if not statements:
        return
    result = None
    for stmt in statements:
        cursor = conn.execute(stmt)
        if cursor.description:
            result = cursor.fetchall()
            columns = [d[0] for d in cursor.description]
    conn.commit()
    if result is None:
        print('✅ Done')
        return
    if not result:
        print('(empty)')
        return
    col_widths = [len(c) for c in columns]
    for row in result:
        for i, val in enumerate(row):
            col_widths[i] = max(col_widths[i], len(str(val)))
    header = ' | '.join(c.ljust(col_widths[i]) for i, c in enumerate(columns))
    sep = '-+-'.join('-' * col_widths[i] for i in range(len(columns)))
    print(header)
    print(sep)
    for row in result:
        print(' | '.join(str(v).ljust(col_widths[i]) for i, v in enumerate(row)))
    print(f'\n({len(result)} row{"s" if len(result) != 1 else ""})')

def describe(table_name):
    print(f'\n📋 {table_name} 表结构：')
    print(f'   {"列名":15s} {"类型":10s} {"约束"}')
    print(f'   {"-"*15} {"-"*10} {"-"*20}')
    for row in conn.execute(f"PRAGMA table_info('{table_name}')"):
        constraints = []
        if row[3]: constraints.append('NOT NULL')
        if row[5]: constraints.append('PRIMARY KEY')
        constr = ', '.join(constraints) if constraints else ''
        print(f'   {row[1]:15s} {row[2]:10s} {constr}')
    fks = conn.execute(f"PRAGMA foreign_key_list('{table_name}')").fetchall()
    for fk in fks:
        print(f'   → FK: {fk[3]} → {fk[2]}({fk[4]})  ON DELETE {fk[6]}')


conn = sqlite3.connect('school.db')
conn.execute("PRAGMA foreign_keys = ON")

# ========== 建表：学校管理系统 ==========
conn.executescript('''
DROP TABLE IF EXISTS prerequisites;
DROP TABLE IF EXISTS enrollments;
DROP TABLE IF EXISTS audit_log;
DROP TABLE IF EXISTS courses;
DROP TABLE IF EXISTS students;
DROP TABLE IF EXISTS teachers;

CREATE TABLE teachers (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    department TEXT,
    hire_date TEXT DEFAULT (date('now')),
    deleted_at TEXT DEFAULT NULL
);

CREATE TABLE students (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE NOT NULL,
    enrolled_year INTEGER DEFAULT 2024
);

CREATE TABLE courses (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    code TEXT UNIQUE,
    credits INTEGER DEFAULT 3 CHECK (credits > 0),
    teacher_id INTEGER,
    deleted_at TEXT DEFAULT NULL,
    FOREIGN KEY (teacher_id) REFERENCES teachers(id) ON DELETE SET NULL
);

CREATE TABLE enrollments (
    id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    grade REAL CHECK (grade >= 0 AND grade <= 100),
    enrolled_at TEXT DEFAULT (date('now')),
    FOREIGN KEY (student_id) REFERENCES students(id) ON DELETE CASCADE,
    FOREIGN KEY (course_id) REFERENCES courses(id) ON DELETE CASCADE,
    UNIQUE(student_id, course_id)
);

CREATE TABLE prerequisites (
    id INTEGER PRIMARY KEY,
    course_id INTEGER NOT NULL,
    prereq_id INTEGER NOT NULL,
    FOREIGN KEY (course_id) REFERENCES courses(id) ON DELETE CASCADE,
    FOREIGN KEY (prereq_id) REFERENCES courses(id) ON DELETE CASCADE,
    UNIQUE(course_id, prereq_id)
);

CREATE TABLE audit_log (
    id INTEGER PRIMARY KEY,
    table_name TEXT NOT NULL,
    action TEXT NOT NULL,
    record_id INTEGER,
    old_data TEXT,
    new_data TEXT,
    changed_at TEXT DEFAULT (datetime('now'))
);
''')

# ========== 填充数据 ==========
conn.executescript('''
INSERT INTO teachers (name, email, department, hire_date) VALUES
    ('Li Wei',   'liwei@school.edu',   'Computer Science', '2020-09-01'),
    ('Wang Fang','wangfang@school.edu','Mathematics',      '2019-03-15'),
    ('Zhang Ming','zhangming@school.edu','Computer Science','2021-09-01'),
    ('Chen Xue', 'chenxue@school.edu', 'English',          '2018-09-01');

INSERT INTO students (name, email, enrolled_year) VALUES
    ('Zhao Lei',  'zhaolei@stu.edu',  2023),
    ('Qian Mei',  'qianmei@stu.edu',  2023),
    ('Sun Hao',   'sunhao@stu.edu',   2024),
    ('Li Juan',   'lijuan@stu.edu',   2024),
    ('Zhou Jie',  'zhoujie@stu.edu',  2023),
    ('Wu Feng',   'wufeng@stu.edu',   2024),
    ('Zheng Yu',  'zhengyu@stu.edu',  2023),
    ('Wang Xin',  'wangxin@stu.edu',  2024);

INSERT INTO courses (name, code, credits, teacher_id) VALUES
    ('Python 入门',   'CS101',  4, 1),
    ('数据库基础',     'CS201',  3, 3),
    ('机器学习',       'CS301',  4, 1),
    ('高等数学',       'MATH101',5, 2),
    ('算法设计',       'CS202',  3, 3),
    ('英语写作',       'ENG101', 2, 4);

INSERT INTO prerequisites (course_id, prereq_id) VALUES
    (2, 1),
    (5, 1),
    (3, 4),
    (5, 2);

INSERT INTO enrollments (student_id, course_id, grade, enrolled_at) VALUES
    (1, 1, 92.5, '2024-02-15'), (1, 2, 88.0, '2024-02-15'),
    (1, 4, 76.0, '2024-02-16'), (1, 6, 85.5, '2024-02-17'),
    (2, 1, 95.0, '2024-02-15'), (2, 3, 91.0, '2024-02-18'),
    (2, 4, 82.5, '2024-02-16'),
    (3, 1, 78.0, '2024-02-15'), (3, 2, 85.0, '2024-02-15'),
    (3, 5, 90.0, '2024-02-19'),
    (4, 1, 88.5, '2024-02-15'), (4, 4, 93.0, '2024-02-16'),
    (4, 6, 72.0, '2024-02-17'),
    (5, 2, 91.0, '2024-02-15'), (5, 3, 87.5, '2024-02-18'),
    (5, 5, 94.0, '2024-02-19'),
    (6, 1, 65.0, '2024-02-15'), (6, 2, 70.5, '2024-02-15'),
    (6, 6, 80.0, '2024-02-17'),
    (7, 3, 96.0, '2024-02-18'), (7, 4, 89.0, '2024-02-16'),
    (7, 5, 88.0, '2024-02-19'),
    (8, 6, 77.5, '2024-02-17');
''')

conn.commit()
print('✅ 学校数据库就绪！')
print(f'   teachers:   {conn.execute("SELECT COUNT(*) FROM teachers").fetchone()[0]} 人')
print(f'   students:   {conn.execute("SELECT COUNT(*) FROM students").fetchone()[0]} 人')
print(f'   courses:    {conn.execute("SELECT COUNT(*) FROM courses").fetchone()[0]} 门')
print(f'   enrollments:{conn.execute("SELECT COUNT(*) FROM enrollments").fetchone()[0]} 条')
print(f'   prerequisites: {conn.execute("SELECT COUNT(*) FROM prerequisites").fetchone()[0]} 条')
print(f'   audit_log:  {conn.execute("SELECT COUNT(*) FROM audit_log").fetchone()[0]} 条（初始为空）')


In [ ]:
# 快速预览各表
for t in ['teachers','students','courses','enrollments','prerequisites']:
    describe(t)


---
## 5.1 触发器（Triggers）

触发器 = 当某表发生 INSERT / UPDATE / DELETE 时，**自动执行**的 SQL 代码。

```
CREATE TRIGGER 名字
BEFORE/AFTER INSERT/UPDATE/DELETE ON 表名
FOR EACH ROW
BEGIN
    -- 自动执行的语句
END;
```

核心概念：
- `NEW.列名` → INSERT 和 UPDATE 时，新行的值
- `OLD.列名` → UPDATE 和 DELETE 时，旧行的值
- `BEFORE` → 操作前触发（可修改 NEW 值、可阻止操作）
- `AFTER` → 操作后触发（写日志、级联更新）

In [ ]:
# 触发器 ①：自动记录审计日志 — 学生信息变更时写入 audit_log
sql('''
CREATE TRIGGER audit_student_update
AFTER UPDATE ON students
FOR EACH ROW
BEGIN
    INSERT INTO audit_log (table_name, action, record_id, old_data, new_data, changed_at)
    VALUES ('students', 'UPDATE', NEW.id,
            'name=' || OLD.name || ', email=' || OLD.email,
            'name=' || NEW.name || ', email=' || NEW.email,
            datetime('now'));
END;
''')

# 测试：修改学生邮箱
print('修改前：')
sql("SELECT id, name, email FROM students WHERE id = 1;")

sql("UPDATE students SET email = 'zhaolei_new@stu.edu' WHERE id = 1;")

print('查看审计日志：')
sql('SELECT * FROM audit_log;')


In [ ]:
# 触发器 ②：数据验证 — 阻止非法成绩（BEFORE INSERT + BEFORE UPDATE）
sql('''
CREATE TRIGGER validate_grade_insert
BEFORE INSERT ON enrollments
FOR EACH ROW
BEGIN
    SELECT CASE
        WHEN NEW.grade < 0 OR NEW.grade > 100 THEN
            RAISE(ABORT, '❌ 成绩必须在 0-100 之间！')
    END;
END;
''')

sql('''
CREATE TRIGGER validate_grade_update
BEFORE UPDATE ON enrollments
FOR EACH ROW
BEGIN
    SELECT CASE
        WHEN NEW.grade < 0 OR NEW.grade > 100 THEN
            RAISE(ABORT, '❌ 成绩必须在 0-100 之间！')
    END;
END;
''')

print('测试非法成绩（120）：')
try:
    sql("INSERT INTO enrollments (student_id, course_id, grade) VALUES (8, 3, 120);")
except Exception as e:
    print(f'触发器阻止了: {e}')

print('\n测试合法成绩（87）：')
sql("INSERT INTO enrollments (student_id, course_id, grade) VALUES (8, 3, 87);")
sql("SELECT s.name, c.name AS course, e.grade FROM enrollments e JOIN students s ON e.student_id = s.id JOIN courses c ON e.course_id = c.id WHERE e.student_id = 8;")


In [ ]:
# 触发器 ③：防止误删 — 有学生选修的课程不能删
sql('''
CREATE TRIGGER prevent_course_delete
BEFORE DELETE ON courses
FOR EACH ROW
WHEN (SELECT COUNT(*) FROM enrollments WHERE course_id = OLD.id) > 0
BEGIN
    SELECT RAISE(ABORT, '❌ 课程 "' || OLD.name || '" 还有 ' ||
        (SELECT COUNT(*) FROM enrollments WHERE course_id = OLD.id) ||
        ' 名学生选修，不能删除！');
END;
''')

print('尝试删除 Python 入门（id=1，有很多学生选修）...')
try:
    sql("DELETE FROM courses WHERE id = 1;")
except Exception as e:
    print(f'触发器阻止了: {e}')


In [ ]:
# 查看所有已创建的触发器
sql("SELECT name, tbl_name FROM sqlite_master WHERE type = 'trigger';")


### ✏️ 自己动手

写一个触发器 `audit_student_delete`：当学生被删除时，记录到 audit_log（action='DELETE'，old_data 包含学生姓名和邮箱）。

In [ ]:
sql("""
-- 在这里写触发器：CREATE TRIGGER audit_student_delete ...

-- 测试：
-- DELETE FROM students WHERE id = 8;
-- SELECT * FROM audit_log WHERE action = 'DELETE';
""")


---
## 5.2 视图（Views）

视图 = 一个被保存的 SELECT 查询，可以像表一样使用。**视图不存数据**，每次查询时动态执行。

In [ ]:
# 视图 ①：课程目录 — courses + teachers JOIN 封装成视图
sql('''
CREATE VIEW course_catalog AS
SELECT
    c.id,
    c.code,
    c.name,
    c.credits,
    t.name AS teacher,
    t.department
FROM courses c
LEFT JOIN teachers t ON c.teacher_id = t.id
WHERE c.deleted_at IS NULL;
''')

print('✅ 视图 course_catalog 创建完成')
print('\n用法和普通表一模一样：')
sql('SELECT * FROM course_catalog ORDER BY department, code;')
print('\n还可以在视图上继续筛选：')
sql("SELECT name, teacher, credits FROM course_catalog WHERE department = 'Computer Science';")


In [ ]:
# 视图 ②：学生成绩单 — 带等级转换（CASE WHEN）
sql('''
CREATE VIEW student_transcript AS
SELECT
    s.name AS student,
    c.code,
    c.name AS course,
    e.grade,
    CASE
        WHEN e.grade >= 90 THEN 'A'
        WHEN e.grade >= 80 THEN 'B'
        WHEN e.grade >= 70 THEN 'C'
        WHEN e.grade >= 60 THEN 'D'
        ELSE 'F'
    END AS letter_grade
FROM students s
JOIN enrollments e ON s.id = e.student_id
JOIN courses c ON e.course_id = c.id
WHERE c.deleted_at IS NULL
ORDER BY s.name, c.code;
''')

sql("SELECT * FROM student_transcript WHERE student = 'Zhao Lei';")
print('\n💡 所有 JOIN + CASE 逻辑被封装进一个视图中，查起来超简单')


In [ ]:
# 视图 ③：课程统计 — 每门课的平均分、人数、最高最低
sql('''
CREATE VIEW course_summary AS
SELECT
    c.code,
    c.name,
    COUNT(e.student_id) AS student_count,
    ROUND(AVG(e.grade), 1) AS avg_grade,
    MAX(e.grade) AS highest,
    MIN(e.grade) AS lowest
FROM courses c
LEFT JOIN enrollments e ON c.id = e.course_id
WHERE c.deleted_at IS NULL
GROUP BY c.id, c.code, c.name;
''')

sql('SELECT * FROM course_summary ORDER BY avg_grade DESC;')


### ✏️ 自己动手

创建一个视图 `cs_courses`，只显示 Computer Science 系的课程（用 `course_catalog` 视图）。

In [ ]:
sql("""
-- CREATE VIEW cs_courses AS ...
""")


---
## 5.3 CTE（Common Table Expressions）

CTE = 用 `WITH` 定义临时命名查询，只在一条 SQL 内有效。从上往下读，像写程序一样。

In [ ]:
# CTE ①：简单 WITH — 把子查询提到前面，按名字引用
sql('''
WITH student_avg AS (
    SELECT student_id, ROUND(AVG(grade), 1) AS avg_grade
    FROM enrollments
    GROUP BY student_id
)
SELECT s.name, sa.avg_grade
FROM students s
JOIN student_avg sa ON s.id = sa.student_id
ORDER BY sa.avg_grade DESC;
''')
print('对比：如果没有 CTE，就需要嵌套子查询，CTE 让逻辑从上往下流')


In [ ]:
# CTE ②：多 CTE 链式组合 — 分步计算，每一步引用前面的结果
sql('''
WITH
-- Step 1: 每门课的统计
course_stats AS (
    SELECT course_id, COUNT(*) AS cnt, ROUND(AVG(grade), 1) AS avg_g
    FROM enrollments
    GROUP BY course_id
),
-- Step 2: 只留下"热门课"（≥ 3 人选）
popular_courses AS (
    SELECT * FROM course_stats WHERE cnt >= 3
),
-- Step 3: 找这些热门课中平均分最高的
top_course AS (
    SELECT * FROM popular_courses
    WHERE avg_g = (SELECT MAX(avg_g) FROM popular_courses)
)
-- 最终：关联课程名
SELECT c.name, c.code, tc.cnt AS students, tc.avg_g AS avg_grade
FROM top_course tc
JOIN courses c ON tc.course_id = c.id;
''')
print('\n💡 3 步 CTE，从上往下读，每步都清晰可理解')


In [ ]:
# CTE ③：递归 CTE — 生成数字序列
sql('''
WITH RECURSIVE counter(n) AS (
    SELECT 1                     -- ① 基础：从 1 开始
    UNION ALL
    SELECT n + 1 FROM counter    -- ② 递归：每个数 +1
    WHERE n < 10                 -- ③ 停止条件
)
SELECT * FROM counter;
''')
print('\n递归 CTE 三部曲：基础查询 → UNION ALL → 递归查询（引用自身）+ 终止条件')


In [ ]:
# CTE ④：实战递归 — 找「算法设计(CS202)」的所有前置课程链
# 前置：CS202 ← CS101 + CS201, CS201 ← CS101
#        CS301 ← MATH101

sql('''
WITH RECURSIVE prereq_chain(id, name, code, depth) AS (
    -- 基础：CS202 的直接前置课
    SELECT p.prereq_id, c.name, c.code, 1
    FROM prerequisites p
    JOIN courses c ON p.prereq_id = c.id
    WHERE p.course_id = (SELECT id FROM courses WHERE code = 'CS202')
    UNION ALL
    -- 递归：前置课的前置课
    SELECT p.prereq_id, c.name, c.code, pc.depth + 1
    FROM prerequisites p
    JOIN courses c ON p.prereq_id = c.id
    JOIN prereq_chain pc ON p.course_id = pc.id
    WHERE pc.depth < 10
)
SELECT DISTINCT code, name, MIN(depth) AS min_depth
FROM prereq_chain
GROUP BY code, name
ORDER BY min_depth;
''')
print('\n说明：算法设计(CS202) 的直接前置是 CS101 和 CS201')
print('       数据库基础(CS201) 又依赖 CS101')
print('       所以 CS101 出现在深度 1 和 2 中（取最小深度 = 1）')


### ✏️ 自己动手

用递归 CTE 生成 1 到 20 的斐波那契数列。

> 参考思路：
> ```sql
> WITH RECURSIVE fib(step, a, b) AS (
>     SELECT 1, 0, 1    -- step, fib(n-1), fib(n)
>     UNION ALL
>     SELECT step + 1, b, a + b FROM fib WHERE step < 20
> )
> SELECT step, b AS fib FROM fib;
> ```

In [ ]:
sql("""
-- 在这里写斐波那契递归 CTE
""")


---
## 5.4 软删除（Soft Deletes）

硬删除 = 数据永久消失。软删除 = 标记 `deleted_at`，数据还在，可恢复。

我们的 `courses` 和 `teachers` 表已经建好了 `deleted_at` 列，直接演示。

In [ ]:
# 软删除：把英语写作(ENG101) 标记为已删除
print('删除前——所有课程：')
sql('SELECT id, code, name, deleted_at FROM courses;')

sql("UPDATE courses SET deleted_at = datetime('now') WHERE code = 'ENG101';")

print('\n软删除后——ENG101 的 deleted_at 有值了：')
sql('SELECT id, code, name, deleted_at FROM courses;')

print('\n查询"活跃"课程（deleted_at IS NULL）—— ENG101 不出现了：')
sql('SELECT id, code, name FROM courses WHERE deleted_at IS NULL;')


In [ ]:
# 恢复软删除——一行 UPDATE 就搞定
sql("UPDATE courses SET deleted_at = NULL WHERE code = 'ENG101';")
print('恢复后：')
sql("SELECT id, code, name, deleted_at FROM courses WHERE code = 'ENG101';")
print('\n✅ 硬删除的话只能从备份恢复，软删除一行 UPDATE 就恢复了')


In [ ]:
# 最佳实践：视图 + 软删除 —— 应用层只查视图，自动过滤已删除数据
sql('''
CREATE VIEW IF NOT EXISTS active_courses AS
SELECT id, code, name, credits, teacher_id
FROM courses
WHERE deleted_at IS NULL;
''')

# 软删 ENG101
sql("UPDATE courses SET deleted_at = datetime('now') WHERE code = 'ENG101';")

print('查原表 courses（ENG101 还在，但 deleted_at 有值）：')
sql('SELECT name, deleted_at FROM courses;')
print('\n查 active_courses 视图（自动过滤已删除）：')
sql('SELECT name FROM active_courses;')

# 恢复以便后续演示
sql("UPDATE courses SET deleted_at = NULL WHERE code = 'ENG101';")


### ✏️ 自己动手

软删除教师 `Chen Xue`（id=4），然后创建 `active_teachers` 视图来过滤已删教师。

In [ ]:
sql("""
-- CREATE VIEW active_teachers AS SELECT * FROM teachers WHERE deleted_at IS NULL;
-- UPDATE teachers SET deleted_at = datetime('now') WHERE name = 'Chen Xue';
-- SELECT * FROM active_teachers;
""")


---
## 5.5 多对多关系（Many-to-Many）

一个学生选多门课，一门课有多个学生 → N:M。通过联结表 `enrollments` 实现。

```
students ──┬── enrollments ──┬── courses
   1:N         (联结表)        N:1
```

In [ ]:
# N:M 基本查询：三表 JOIN，列出所有学生的选课
sql('''
SELECT s.name AS student, c.code, c.name AS course, e.grade, e.enrolled_at
FROM students s
JOIN enrollments e ON s.id = e.student_id
JOIN courses c ON e.course_id = c.id
ORDER BY s.name, c.code;
''')
print('↑ 三表 JOIN: students → enrollments → courses')


In [ ]:
# N:M 聚合：每门课有多少人选、平均分、学生名单
sql('''
SELECT c.code, c.name,
       COUNT(e.student_id) AS student_count,
       ROUND(AVG(e.grade), 1) AS avg_grade,
       GROUP_CONCAT(s.name, ', ') AS students
FROM courses c
LEFT JOIN enrollments e ON c.id = e.course_id
LEFT JOIN students s ON e.student_id = s.id
WHERE c.deleted_at IS NULL
GROUP BY c.id
ORDER BY student_count DESC, avg_grade DESC;
''')
print('↑ GROUP_CONCAT 把多行姓名合并成一行，方便一览每门课的学生')


In [ ]:
# N:M 进阶：找出同时选了 Python 入门(CS101) 和 数据库基础(CS201) 的学生
sql('''
SELECT s.name
FROM students s
JOIN enrollments e1 ON s.id = e1.student_id
JOIN enrollments e2 ON s.id = e2.student_id
JOIN courses c1 ON e1.course_id = c1.id
JOIN courses c2 ON e2.course_id = c2.id
WHERE c1.code = 'CS101' AND c2.code = 'CS201';
''')
print('↑ 同一个 students 表 JOIN 两次 enrollments，分别匹配两门课')


In [ ]:
# N:M 进阶：选了 Python 入门(CS101) 但没选 数据库基础(CS201) 的学生
sql('''
SELECT s.name
FROM students s
JOIN enrollments e ON s.id = e.student_id
JOIN courses c ON e.course_id = c.id
WHERE c.code = 'CS101'
  AND s.id NOT IN (
      SELECT student_id FROM enrollments e2
      JOIN courses c2 ON e2.course_id = c2.id
      WHERE c2.code = 'CS201'
  );
''')
print('↑ 在 CS101 中但不在 CS201 中的学生')


In [ ]:
# N:M 联结表 = N:M 关系 + 关系本身的属性
print('enrollments 表结构：两个 FK(student_id, course_id) + 附加属性(grade, enrolled_at)')
describe('enrollments')
print('\n数据预览：')
sql('SELECT * FROM enrollments ORDER BY enrolled_at, student_id LIMIT 10;')
print('\n💡 联结表的 grade 和 enrolled_at 描述的是"选课"这个关系本身的属性')


### ✏️ 自己动手

找出选了 3 门及以上课程的学生名单和选课数量。

In [ ]:
sql("""
-- 在这里写查询（提示：GROUP BY student_id, HAVING COUNT(*) >= 3）
""")


---
## 🎯 综合挑战

In [ ]:
# Q1：创建一个触发器 log_enrollment，当 enrollments 表 INSERT 时
#     自动记录到 audit_log（action='ENROLL'，new_data 包含 student_id 和 course_id）
sql("""
-- CREATE TRIGGER log_enrollment ...
-- 写完后测试：INSERT INTO enrollments (student_id, course_id, grade) VALUES (8, 5, 82);
-- 然后查 audit_log
""")


In [ ]:
# Q2：创建视图 honor_roll，列出平均分 ≥ 90 的学生（姓名 + 平均分 + 选课数）
sql("""
-- CREATE VIEW honor_roll AS ...
""")


In [ ]:
# Q3：用递归 CTE 计算 5 的阶乘（5! = 5×4×3×2×1 = 120）
# 提示：step 从 1 递增到 5，result 累乘
sql("""
-- WITH RECURSIVE factorial(step, result) AS (
--     SELECT 1, 1
--     UNION ALL
--     SELECT step + 1, result * (step + 1) FROM factorial WHERE step < 5
-- )
-- SELECT * FROM factorial;
""")
print('提示：去掉注释即可运行')


In [ ]:
# Q4：软删除一门有选课记录的课程，观察 enrollments 中的相关记录还在不在。
#     软删除不会 CASCADE——想想这是好是坏？什么时候该用硬删除？
sql("""
-- UPDATE courses SET deleted_at = datetime('now') WHERE code = '...';
-- SELECT e.* FROM enrollments e JOIN courses c ON e.course_id = c.id WHERE c.code = '...';
""")


In [ ]:
# Q5：用 CTE + ROW_NUMBER() 窗口函数找出「每个学生成绩最高的那门课」
#     提示：SQLite 3.25+ 支持窗口函数
sql("""
-- 如果你的 SQLite 版本 >= 3.25，可以这样：
-- WITH ranked AS (
--     SELECT s.name AS student, c.name AS course, e.grade,
--            ROW_NUMBER() OVER (PARTITION BY s.id ORDER BY e.grade DESC) AS rn
--     FROM students s
--     JOIN enrollments e ON s.id = e.student_id
--     JOIN courses c ON e.course_id = c.id
-- )
-- SELECT student, course, grade FROM ranked WHERE rn = 1;
""")
print('提示：去掉注释即可运行（SQLite 3.25+）')
print('如果你的 SQLite 版本较低，可以用相关子查询替代 ROW_NUMBER()')


---
## ✅ 检查清单

- [ ] 理解什么是触发器，能写出 BEFORE/AFTER INSERT/UPDATE/DELETE 触发器
- [ ] 理解 OLD 和 NEW 的引用场景（INSERT 有 NEW，DELETE 有 OLD，UPDATE 两者都有）
- [ ] 能用 RAISE(ABORT, ...) 在触发器中阻止非法操作
- [ ] 能用视图封装复杂 JOIN，简化日常查询
- [ ] 理解视图是虚拟表，不存数据
- [ ] 能用 WITH 定义 CTE，从上往下组织查询逻辑
- [ ] 理解递归 CTE 的三部曲：基础 + UNION ALL + 递归（加终止条件）
- [ ] 能用递归 CTE 解决层级数据问题（课程依赖、组织架构、目录树）
- [ ] 理解软删除 vs 硬删除的取舍，知道何时用哪个
- [ ] 能用 deleted_at 列 + 视图实现软删除
- [ ] 能设计联结表解决多对多关系
- [ ] 能在联结表上做聚合查询（COUNT、AVG、GROUP_CONCAT）
- [ ] 能写同时查多个条件的 N:M 查询（同时选了两门课的学生等）

---

> 📖 下一章：[06 - 性能与事务](../06-performance-and-transactions/) — Index、ACID、并发控制